# Final Model Improvement Experiments

This notebook investigates whether the predictive performance of the
selected forest-fire burned-area model can be improved further.

The current final model is a weighted ensemble consisting of:

- 70% HistGradientBoosting Regressor
- 30% Random Forest Regressor

The baseline model achieved:

- Test R²: 0.2302
- Test MAE: 0.8328
- Test RMSE: 1.0421

Although this model provides the strongest performance obtained so far,
previous error analysis showed substantial underprediction of rare
extreme-fire events.

Therefore, this notebook investigates alternative modelling strategies,
including advanced gradient boosting, alternative target transformations,
cross-validation, and severity-aware weighting.

The existing final model is retained as the baseline and will not be
modified.

The objective is to determine whether a statistically and scientifically
meaningful improvement can be achieved without introducing data leakage
or excessive model complexity.

In [1]:
# IMPORT LIBRARIES

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor
)

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# LOAD PROCESSED DATA

PROJECT_ROOT = Path("../../")
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"

X_train = pd.read_csv(
    PROCESSED_DIR / "X_train.csv"
)

X_test = pd.read_csv(
    PROCESSED_DIR / "X_test.csv"
)

y_train = pd.read_csv(
    PROCESSED_DIR / "y_train.csv"
).squeeze("columns")

y_test = pd.read_csv(
    PROCESSED_DIR / "y_test.csv"
).squeeze("columns")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (9042, 44)
X_test : (2261, 44)
y_train: (9042,)
y_test : (2261,)


In [3]:
# LOAD CURRENT BEST MODEL

MODEL_DIR = PROJECT_ROOT / "models"

baseline_hist = joblib.load(
    MODEL_DIR / "hist_gradient_boosting_final.pkl"
)

baseline_rf = joblib.load(
    MODEL_DIR / "random_forest_final.pkl"
)

baseline_hist_pred = baseline_hist.predict(X_test)
baseline_rf_pred = baseline_rf.predict(X_test)

baseline_pred = (
    0.70 * baseline_hist_pred
    + 0.30 * baseline_rf_pred
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_pred
    )
)

print("=" * 55)
print("CURRENT BASELINE MODEL")
print("=" * 55)

print(f"R²   : {baseline_r2:.4f}")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")

CURRENT BASELINE MODEL
R²   : 0.2302
MAE  : 0.8328
RMSE : 1.0421


## XGBoost

In [5]:
try:
    import xgboost as xgb

    print("XGBoost version:", xgb.__version__)
    print("XGBoost is available.")

except ImportError:
    print("XGBoost is NOT installed in the current environment.")

XGBoost version: 3.4.0
XGBoost is available.


In [6]:
# XGBOOST BASELINE MODEL

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.0


In [7]:
# TRAIN XGBOOST BASELINE

xgb_baseline = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")

xgb_baseline.fit(
    X_train,
    y_train
)

print("XGBoost training completed.")

Training XGBoost...
XGBoost training completed.


In [8]:
# EVALUATE XGBOOST


xgb_train_pred = xgb_baseline.predict(X_train)
xgb_test_pred = xgb_baseline.predict(X_test)

xgb_train_r2 = r2_score(
    y_train,
    xgb_train_pred
)

xgb_test_r2 = r2_score(
    y_test,
    xgb_test_pred
)

xgb_test_mae = mean_absolute_error(
    y_test,
    xgb_test_pred
)

xgb_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_test_pred
    )
)

print("=" * 55)
print("XGBOOST BASELINE RESULTS")
print("=" * 55)

print(f"Train R² : {xgb_train_r2:.4f}")
print(f"Test R²  : {xgb_test_r2:.4f}")
print(f"Test MAE : {xgb_test_mae:.4f}")
print(f"Test RMSE: {xgb_test_rmse:.4f}")

print()
print("CURRENT ENSEMBLE BASELINE")
print("=" * 55)

print(f"Test R²  : {baseline_r2:.4f}")
print(f"Test MAE : {baseline_mae:.4f}")
print(f"Test RMSE: {baseline_rmse:.4f}")

XGBOOST BASELINE RESULTS
Train R² : 0.6931
Test R²  : 0.2356
Test MAE : 0.8284
Test RMSE: 1.0385

CURRENT ENSEMBLE BASELINE
Test R²  : 0.2302
Test MAE : 0.8328
Test RMSE: 1.0421


### Interpretation

The baseline XGBoost model achieved a test R² of 0.2356, meaning it explains approximately 23.6% of the variation in the log-transformed burned-area target on unseen data. This represents a small improvement over the existing 70% HistGradientBoosting + 30% Random Forest ensemble, which achieved a test R² of 0.2302.

The XGBoost model also produced slightly lower errors, with a Test MAE of 0.8284 and Test RMSE of 1.0385, compared with 0.8328 and 1.0421 for the existing ensemble. Therefore, XGBoost currently provides the best individual model performance observed so far, although the improvement is relatively small.

The training R² of 0.6931 is considerably higher than the test R² of 0.2356, indicating a noticeable generalisation gap. This suggests that XGBoost is learning substantial patterns from the training data but is unable to reproduce all of those patterns reliably on unseen observations. This is consistent with the dataset's challenging characteristics, particularly its highly skewed burned-area distribution and relatively limited representation of extreme fire events.

### XGBoost Hyperparameter Tuning

In [9]:
import optuna

print("Optuna version:", optuna.__version__)
print("Optuna loaded successfully.")

Optuna version: 4.9.0
Optuna loaded successfully.


In [10]:
# XGBOOST ADVANCED TUNING SETUP

import optuna
import xgboost as xgb
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

print("XGBoost version:", xgb.__version__)
print("Optuna version:", optuna.__version__)

print("Advanced tuning environment ready.")

XGBoost version: 3.4.0
Optuna version: 4.9.0
Advanced tuning environment ready.


In [11]:
# ADVANCED XGBOOST HYPERPARAMETER OPTIMIZATION

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# 5-fold cross-validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            300,
            1500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.08,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            30
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.50,
            1.00
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            2.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.01,
            20.0,
            log=True
        ),

        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1
    }

    fold_scores = []

    for train_idx, val_idx in kf.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        val_pred = model.predict(X_val)

        fold_r2 = r2_score(
            y_val,
            val_pred
        )

        fold_scores.append(fold_r2)

    return np.mean(fold_scores)


# Create Optuna study
study = optuna.create_study(
    direction="maximize",
    study_name="forest_fire_xgboost_advanced"
)

print("=" * 60)
print("STARTING ADVANCED XGBOOST OPTIMIZATION")
print("=" * 60)
print("Cross-validation folds: 5")
print("Optimization trials: 150")
print()
print("This may take some time...")


study.optimize(
    objective,
    n_trials=150,
    show_progress_bar=True
)

print()
print("=" * 60)
print("OPTIMIZATION COMPLETED")
print("=" * 60)

[I 2026-08-10 14:58:41,857] A new study created in memory with name: forest_fire_xgboost_advanced


STARTING ADVANCED XGBOOST OPTIMIZATION
Cross-validation folds: 5
Optimization trials: 150

This may take some time...


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-08-10 14:59:34,641] Trial 0 finished with value: 0.19096844660973758 and parameters: {'n_estimators': 1442, 'learning_rate': 0.032340801666427436, 'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.6372122738001602, 'colsample_bytree': 0.6850606094209768, 'gamma': 0.5098648301766942, 'reg_alpha': 0.0009240914237750821, 'reg_lambda': 4.865372846352219}. Best is trial 0 with value: 0.19096844660973758.
[I 2026-08-10 15:00:24,907] Trial 1 finished with value: 0.18847488311573293 and parameters: {'n_estimators': 1391, 'learning_rate': 0.009692572977365983, 'max_depth': 3, 'min_child_weight': 25, 'subsample': 0.7082603751724512, 'colsample_bytree': 0.6697988567764375, 'gamma': 1.6553574301633909, 'reg_alpha': 4.4150906911786825, 'reg_lambda': 6.224772000780406}. Best is trial 0 with value: 0.19096844660973758.
[I 2026-08-10 15:02:43,927] Trial 2 finished with value: 0.20842565868966045 and parameters: {'n_estimators': 1265, 'learning_rate': 0.005308711257730296, 'max_depth': 9, 

In [12]:
# BEST XGBOOST CONFIGURATION

print("=" * 60)
print("BEST XGBOOST TUNING RESULT")
print("=" * 60)

print("Best 5-Fold CV R²:")
print(round(study.best_value, 5))

print("\nBest Parameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

BEST XGBOOST TUNING RESULT
Best 5-Fold CV R²:
0.21235

Best Parameters:
n_estimators: 1268
learning_rate: 0.008276102891538179
max_depth: 7
min_child_weight: 4
subsample: 0.867007492731578
colsample_bytree: 0.5892267514768841
gamma: 1.5964641892367197
reg_alpha: 3.0398616203941947e-05
reg_lambda: 12.90612737170908


### Interpretation

The optimization explored 150 configurations, and the best configuration achieved a cross-validation R² of 0.2124. This indicates that the tuned XGBoost model can learn meaningful relationships in the training data, but the improvement from hyperparameter optimization is limited.

The relatively low learning rate (0.0083) combined with a large number of trees (1,268) suggests that the optimization favoured gradual learning rather than aggressive boosting. The relatively strong reg_lambda value also indicates that additional regularization was useful for controlling model complexity.

Most importantly, the CV result does not indicate that we have found a dramatically better model. This supports our earlier observation that the main limitation is likely not simply the XGBoost hyperparameters, but the characteristics of the burned-area target and the difficulty of predicting rare extreme fires.